In [32]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE


In [33]:
df = pd.read_csv("../data/processed/fraud_features.csv")

df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,merch_lat,merch_long,is_fraud,age,transaction_hours,transaction_day,transaction_month,transaction_weekday,is_night_transaction,merchant_distance
0,2020-06-21 13:05:42,60416207185,fraud_Kutch-Ferry,home,124.66,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,42.945526,-108.530901,0,34,13,21,6,6,0,30.457093
1,2020-06-21 16:25:36,60416207185,fraud_Halvorson Group,misc_pos,78.52,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,42.193130,-108.682054,0,34,16,21,6,6,0,91.942935
2,2020-06-22 07:58:33,60416207185,fraud_Conroy-Cruickshank,gas_transport,65.25,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,43.932724,-109.699794,0,34,7,22,6,0,0,121.857420
3,2020-06-22 15:32:31,60416207185,fraud_Larkin Ltd,kids_pets,87.74,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,43.546064,-109.212939,0,34,15,22,6,0,0,65.414164
4,2020-06-23 12:28:54,60416207185,fraud_Leffler-Goldner,personal_care,148.02,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,42.876538,-109.333220,0,34,12,23,6,1,0,38.311273


In [34]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

print(X.shape)
print(y.shape)

(555719, 28)
(555719,)


In [35]:
merchant_map = {
    merchant: idx
    for idx, merchant in enumerate(X["merchant"].unique())
}

X["merchant"] = X["merchant"].map(merchant_map)

In [36]:
os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    merchant_map,
    "../artifacts/merchant_map.joblib"
)

['../artifacts/merchant_map.joblib']

In [37]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

category_encoded = encoder.fit_transform(X[["category"]])

In [38]:
category_df = pd.DataFrame(
    category_encoded,
    columns=encoder.get_feature_names_out(["category"]),
    index=X.index
)
X = pd.concat(
    [
        X.drop(columns=["category"]),
        category_df
    ],
    axis=1
)

In [39]:
joblib.dump(
    encoder,
    "../artifacts/encoder.joblib"
)

['../artifacts/encoder.joblib']

In [40]:
print(X.shape)
print(X.columns.tolist())

(555719, 41)
['trans_date_trans_time', 'cc_num', 'merchant', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'age', 'transaction_hours', 'transaction_day', 'transaction_month', 'transaction_weekday', 'is_night_transaction', 'merchant_distance', 'category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']


In [41]:
drop_columns = [
    "trans_date_trans_time",
    "cc_num",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "unix_time",
    "merch_lat",
    "merch_long"
]

X = X.drop(columns=drop_columns)
print(X.shape)
print(X.columns.tolist())

(555719, 23)
['merchant', 'amt', 'age', 'transaction_hours', 'transaction_day', 'transaction_month', 'transaction_weekday', 'is_night_transaction', 'merchant_distance', 'category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']


In [42]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [43]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(444575, 23)
(111144, 23)
is_fraud
0    442859
1      1716
Name: count, dtype: int64
is_fraud
0    110715
1       429
Name: count, dtype: int64


In [44]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [45]:
import os


os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    scaler,
    "../artifacts/scaler.joblib"
)

['../artifacts/scaler.joblib']

In [46]:
joblib.dump(
    X_train.columns.tolist(),
    "../artifacts/feature_columns.joblib"
)
print(len(X_train.columns))
print(X_train.columns.tolist())

23
['merchant', 'amt', 'age', 'transaction_hours', 'transaction_day', 'transaction_month', 'transaction_weekday', 'is_night_transaction', 'merchant_distance', 'category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel']


In [47]:


smote = SMOTE(random_state=42)
X_train_balanced_scaled, y_train_balanced_scaled = smote.fit_resample(
    X_train_scaled,
    y_train
)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train,
    y_train
)

In [48]:
print("Original Training Labels:")
print(y_train.value_counts())

print("\nBalanced Training Labels:")
print(y_train_balanced.value_counts())

Original Training Labels:
is_fraud
0    442859
1      1716
Name: count, dtype: int64

Balanced Training Labels:
is_fraud
0    442859
1    442859
Name: count, dtype: int64


In [49]:
X_train_balanced.to_csv(
    "../data/processed/X_train_balanced.csv",
    index=False
)

y_train_balanced.to_csv(
    "../data/processed/y_train_balanced.csv",
    index=False
)

X_train_balanced_scaled.to_csv(
    "../data/processed/X_train_balanced_scaled.csv",
    index=False
)

y_train_balanced_scaled.to_csv(
    "../data/processed/y_train_balanced_scaled.csv",
    index=False
)

In [50]:
X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

X_test_scaled.to_csv(
    "../data/processed/X_test_scaled.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)